# Phi-4 Mini Turkish Fine-tune - Kaggle (Internet Yok)
**Platform:** Kaggle T4 GPU  
**Not:** Kaggle'da internet yok, Unsloth dataset'ten kurulur


In [ ]:
# ─── 1. KURULUM (Internet'siz) ──────────────────────────────────────────────
import subprocess, sys, os

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-300:])
    print(result.stdout[-200:])

# Unsloth wheel dataset'ten kur (internet gerekmez)
# Dataset: barnobarno/unsloth-wheels
WHEEL_DIR = '/kaggle/input/unsloth-wheels'

if os.path.exists(WHEEL_DIR):
    print('Unsloth wheel bulundu, kuruluyor...')
    wheels = [f for f in os.listdir(WHEEL_DIR) if f.endswith('.whl')]
    for whl in wheels:
        run(f'pip install --no-index --find-links={WHEEL_DIR} {WHEEL_DIR}/{whl} -q')
    print('OK Unsloth kuruldu')
else:
    print('UYARI: Unsloth wheel dataset bulunamadi!')
    print('Dataset ekleyin: barnobarno/unsloth-wheels')
    print('Veya truthisneverlinear/unsloth-whl')

# Diger paketler (Kaggle'da genelde hazir)
for pkg in ['trl', 'peft', 'accelerate', 'bitsandbytes', 'datasets']:
    try:
        __import__(pkg)
    except ImportError:
        print(f'{pkg} kurulamadi (internet yok)')

print('Kurulum tamamlandi')

In [ ]:
# ─── 2. HF TOKEN ───────────────────────────────────────────────────────────
import os

HF_TOKEN = 'hf_buraya_token_yaz'  # <-- TOKEN BURAYA
os.environ['HF_TOKEN'] = HF_TOKEN
print('HF Token yuklendi' if HF_TOKEN != 'hf_buraya_token_yaz' else 'HF Token bos')

In [ ]:
# ─── 3. MODEL YUKLE ────────────────────────────────────────────────────────
from unsloth import FastLanguageModel
import torch

print(f'CUDA: {torch.cuda.is_available()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Phi-4-mini-instruct',
    max_seq_length=2048,
    load_in_4bit=True,
    token=HF_TOKEN if HF_TOKEN != 'hf_buraya_token_yaz' else None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', use_gradient_checkpointing='unsloth', random_state=42,
)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {total/1e9:.2f}B parametre')
print(f'LoRA: {trainable:,}/{total:,} trainable ({100*trainable/total:.1f}%)')

In [ ]:
# ─── 4. VERI SETLERI ───────────────────────────────────────────────────────
from datasets import load_dataset, Dataset
import random

random.seed(42)
all_data = []

ds_configs = [
    ('merve/turkish_instructions', 'instruction', 'output', 'input'),
    ('ucekmez/OpenOrca-tr', 'question', 'response', None),
    ('atasoglu/databricks-dolly-15k-tr', 'instruction', 'response', 'context'),
    ('ytu-ce-cosmos/gsm8k_tr', 'question', 'answer', None),
    ('beratcmn/no_robots_turkish', 'instruction', 'output', None),
]

for ds_name, col_q, col_a, col_ctx in ds_configs:
    try:
        ds = load_dataset(ds_name, split='train')
        cnt = 0
        for row in ds:
            q = str(row.get(col_q, '')).strip()
            a = str(row.get(col_a, '')).strip()
            if col_ctx and row.get(col_ctx):
                ctx = str(row[col_ctx]).strip()
                if ctx:
                    q = ctx + ' ' + q
            if q and a and len(q) > 5 and len(a) > 5:
                all_data.append({'messages': [
                    {'role': 'user', 'content': q[:1000]},
                    {'role': 'assistant', 'content': a[:1000]}
                ]})
                cnt += 1
                if cnt >= 5000:
                    break
        print(f'  OK {ds_name}: {cnt}')
    except Exception as e:
        print(f'  SKIP {ds_name}: {str(e)[:60]}')

print(f'Toplam {len(all_data)} ornek')

In [ ]:
# ─── 5. EGITIM ─────────────────────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments
import time

def fmt(examples):
    texts = []
    for m in examples['messages']:
        text = tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {'text': texts}

ds = Dataset.from_list(all_data)
ds = ds.map(fmt, batched=True, batch_size=1000)
ds = ds.train_test_split(test_size=0.05, seed=42)
print(f'Train: {len(ds["train"])}, Eval: {len(ds["test"])}')

t0 = time.time()
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=ds['train'], eval_dataset=ds['test'],
    dataset_text_field='text', max_seq_length=2048,
    args=TrainingArguments(
        output_dir='/kaggle/working/output',
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=2e-4, warmup_steps=10,
        logging_steps=50, save_steps=200,
        optim='adamw_8bit', lr_scheduler_type='cosine',
        report_to='none', seed=42, fp16=True,
    ),
)

print('Egitim basliyor...')
result = trainer.train()
elapsed = (time.time() - t0) / 60
print(f'Egitim bitti! Loss: {result.training_loss:.4f}')
print(f'Sure: {elapsed:.1f} dakika')

In [ ]:
# ─── 6. KAYDET & PUSH ──────────────────────────────────────────────────────
model.save_pretrained('/kaggle/working/lora')
tokenizer.save_pretrained('/kaggle/working/lora')
print('Model kaydedildi')

HUB_ID = 'KuroNeko1234t/phi4-mini-tr'

if HF_TOKEN and HF_TOKEN != 'hf_buraya_token_yaz':
    from huggingface_hub import create_repo
    create_repo(HUB_ID, token=HF_TOKEN, exist_ok=True, repo_type='model')
    model.push_to_hub(HUB_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(HUB_ID, token=HF_TOKEN)
    print(f'Push OK: huggingface.co/{HUB_ID}')
else:
    print('HF Token yok, push atlandi')

print('TAMAM!')